# Практическая работа № 4. Регрессионная модель
**Уровень B.** Прогнозируем `tip` (чаевые) по сумме счёта, размеру компании и категориальным признакам. `bill_per_person` создан как дополнительный числовой признак, чтобы проверить мультиколлинеарность; он не использует целевую переменную.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
    roc_curve, precision_recall_curve, roc_auc_score, average_precision_score,
    confusion_matrix, classification_report, precision_score, recall_score, f1_score)
RANDOM_STATE = 42
DATA_PATH = Path("../data/tips.csv") if Path("../data/tips.csv").exists() else Path("data/tips.csv")
df = pd.read_csv(DATA_PATH)
print("Размер:", df.shape)
df.head()

In [ ]:
df = df.copy()
df["bill_per_person"] = df["total_bill"] / df["size"].replace(0, np.nan)
target = "tip"
features = ["total_bill", "size", "bill_per_person", "sex", "smoker", "day", "time"]
X, y = df[features], df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
num_cols = ["total_bill", "size", "bill_per_person"]
cat_cols = ["sex", "smoker", "day", "time"]
preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
])
baseline = Pipeline([("preprocess", preprocessor), ("model", LinearRegression())])
baseline.fit(X_train, y_train)
pred_base = baseline.predict(X_test)
def regression_metrics(y_true, pred):
    return {"MAE": mean_absolute_error(y_true, pred), "RMSE": mean_squared_error(y_true, pred)**0.5, "R2": r2_score(y_true, pred)}
print("Базовая модель:", regression_metrics(y_test, pred_base))

In [ ]:
# Проверка мультиколлинеарности числовых признаков только на train
corr = X_train[num_cols].corr()
display(corr)
plt.figure(figsize=(6,4)); sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Корреляция числовых признаков"); plt.tight_layout(); plt.show()
# Если |corr| >= 0.85, исключаем из пары признак bill_per_person.
reduced_num = ["total_bill", "size"] if abs(corr.loc["total_bill", "bill_per_person"]) >= 0.85 else num_cols
reduced_features = reduced_num + cat_cols
Xr_train, Xr_test = X_train[reduced_features], X_test[reduced_features]
reduced_pre = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), reduced_num),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
])
reduced = Pipeline([("preprocess", reduced_pre), ("model", LinearRegression())])
reduced.fit(Xr_train, y_train)
pred_reduced = reduced.predict(Xr_test)
comparison = pd.DataFrame({"Базовая": regression_metrics(y_test, pred_base), "Сокращённая": regression_metrics(y_test, pred_reduced)}).T
display(comparison)
print("Числовые признаки финальной модели:", reduced_num)

In [ ]:
residuals = y_test - pred_reduced
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(pred_reduced, residuals, alpha=.75)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set(title="Остатки против предсказаний", xlabel="Предсказанные чаевые", ylabel="Остаток")
axes[1].scatter(y_test, pred_reduced, alpha=.75)
lo, hi = min(y_test.min(), pred_reduced.min()), max(y_test.max(), pred_reduced.max())
axes[1].plot([lo, hi], [lo, hi], "r--")
axes[1].set(title="Предсказание vs истина", xlabel="Истинные чаевые", ylabel="Предсказанные чаевые")
plt.tight_layout(); plt.show()
print("MAE означает среднюю абсолютную ошибку в единицах суммы чаевых.")

## Вывод
Сравниваем модели на одном и том же тестовом разбиении. Финальный набор выбирается по компромиссу MAE/RMSE/R² и простоте: если удаление `bill_per_person` не ухудшает метрики заметно, предпочтительнее сокращённая модель. Остатки проверяются на случайное распределение вокруг нуля; видимый паттерн указывает на возможную нелинейность или неодинаковую дисперсию.